# GLADA: Speaker Leakage Probe

**Question:** Does the speaker GRL in GLADA actually remove speaker identity from the bottleneck embedding `z`?

**Protocol:**
1. Load a trained checkpoint (e.g. `full_glada`, `no_speaker`, `backbone_only`).
2. Extract pre-classifier embeddings (`features` = VIB output `z`) on ASVspoof 2019 LA train + dev (same 20 speakers, disjoint utterances).
3. Fit a logistic-regression probe `z -> speaker_id` on train embeddings, evaluate on dev.
4. Compare probe accuracy across configs. Chance = 1/20 = 5%.

**Interpretation:**
- `backbone_only` (no speaker GRL) gives an upper-bound on how much speaker info the SSL backbone naturally encodes.
- `full_glada` (speaker GRL on) — gap to backbone_only = how much the GRL actually disentangled.
- High accuracy on `full_glada` => GRL did not remove speaker info; the negative-transfer result is not from over-aggressive disentanglement.
- Low accuracy on `full_glada` => GRL removed speaker info, possibly at the cost of useful cross-domain features.

Either result directly answers Belman's expected question with data.

## 0. Environment Setup

Run on Colab with the `glada_pkg` synced to Drive (matches the convention used by `glada_evaluation.ipynb`).

In [ ]:
!pip install -q transformers soundfile librosa scikit-learn tqdm

from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/drive/MyDrive/Jakob/glada_pkg')

import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 1. Imports & Paths

In [ ]:
import os, json, time
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, top_k_accuracy_score

from glada.config import GLADAConfig
from glada.models.glada_model import GLADAModel
from glada.datasets.asvspoof import ASVspoofDatasetWithSpeaker

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Paths -- update CHECKPOINTS to match your Drive layout.
CHECKPOINT_ROOT = '/content/drive/MyDrive/Jakob/Experiments'
ASVSPOOF_AUDIO  = '/content/data/asvspoof/LA'
PROTOCOL_TRAIN  = f'{ASVSPOOF_AUDIO}/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt'
AUDIO_TRAIN     = f'{ASVSPOOF_AUDIO}/ASVspoof2019_LA_train/flac'
PROTOCOL_DEV    = f'{ASVSPOOF_AUDIO}/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.dev.trl.txt'
AUDIO_DEV       = f'{ASVSPOOF_AUDIO}/ASVspoof2019_LA_dev/flac'

CACHE_DIR = '/content/embedding_cache'
os.makedirs(CACHE_DIR, exist_ok=True)

# Configs to probe. Each entry: (label, checkpoint_path).
# Edit to match your saved checkpoints. The script will skip any that don't exist.
CHECKPOINTS = [
    ('full_glada',     f'{CHECKPOINT_ROOT}/full_glada/best_model.pt'),
    ('no_speaker',     f'{CHECKPOINT_ROOT}/no_speaker/best_model.pt'),
    ('backbone_only',  f'{CHECKPOINT_ROOT}/backbone_only/best_model.pt'),
]

## 2. Load Checkpoint Helper

Loads a `GLADAModel` from a checkpoint. The checkpoint is expected to contain `state_dict` and `config` (matching the trainer's save format).

In [ ]:
def load_model(checkpoint_path):
    """Load a GLADAModel from a checkpoint. Returns (model, config)."""
    ckpt = torch.load(checkpoint_path, map_location='cpu', weights_only=False)

    # Reconstruct config -- handle both dict and dataclass forms.
    cfg_obj = ckpt.get('config', None)
    if isinstance(cfg_obj, dict):
        config = GLADAConfig(**cfg_obj)
    elif cfg_obj is None:
        # Fall back to default; will likely fail to load if architecture diverges.
        config = GLADAConfig()
    else:
        config = cfg_obj

    model = GLADAModel(config)
    state = ckpt.get('state_dict', ckpt.get('model_state_dict', ckpt))
    missing, unexpected = model.load_state_dict(state, strict=False)
    if missing:
        print(f'  Missing keys: {len(missing)} (first 3): {missing[:3]}')
    if unexpected:
        print(f'  Unexpected keys: {len(unexpected)} (first 3): {unexpected[:3]}')
    model.to(DEVICE).eval()
    return model, config

## 3. Embedding Extraction

Extract the VIB-bottleneck output `z` (`features`) for every utterance in train and dev.

Note: train + dev share the same 20 speakers (different utterances), so a probe trained on train embeddings can be evaluated on dev to measure speaker-identity leakage on unseen utterances of known speakers.

In [ ]:
@torch.no_grad()
def extract_embeddings(model, dataset, batch_size=24):
    """Run the model over a dataset and collect (features, speaker_id, task_label).

    Uses num_workers=0 (Colab DataLoader constraint).
    """
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False,
                        num_workers=0, pin_memory=True)
    feats, spks, labels = [], [], []
    for batch in tqdm(loader, desc='extract'):
        x = batch['input_values'].to(DEVICE, non_blocking=True)
        m = batch['attention_mask'].to(DEVICE, non_blocking=True)
        out = model(x, attention_mask=m)
        feats.append(out['features'].detach().float().cpu().numpy())
        spks.append(batch['speaker_id'].numpy())
        labels.append(batch['task_label'].numpy())
    return (np.concatenate(feats),
            np.concatenate(spks),
            np.concatenate(labels))

def cached_extract(label, split, model, dataset, batch_size=24):
    """Cache embeddings to disk so re-running the probe doesn't re-run inference."""
    cache_path = f'{CACHE_DIR}/{label}_{split}.npz'
    if os.path.exists(cache_path):
        d = np.load(cache_path)
        print(f'  loaded cached embeddings: {cache_path}  ({d["feats"].shape})')
        return d['feats'], d['spks'], d['labels']
    feats, spks, labels = extract_embeddings(model, dataset, batch_size)
    np.savez(cache_path, feats=feats, spks=spks, labels=labels)
    print(f'  saved cache: {cache_path}  ({feats.shape})')
    return feats, spks, labels

## 4. Build Datasets

Use `ASVspoofDatasetWithSpeaker` (no augmentation) so we get speaker labels alongside audio.

In [ ]:
# Use a lightweight config for dataset construction. Backbone choice doesn't
# matter here -- the dataset only handles audio loading and speaker indexing.
probe_cfg = GLADAConfig()  # defaults are fine for max_audio_len etc.

ds_train = ASVspoofDatasetWithSpeaker(PROTOCOL_TRAIN, AUDIO_TRAIN, probe_cfg, augmenter=None)
ds_dev   = ASVspoofDatasetWithSpeaker(PROTOCOL_DEV,   AUDIO_DEV,   probe_cfg, augmenter=None)

print(f'train: {len(ds_train)} samples, {ds_train.num_speakers} speakers')
print(f'dev:   {len(ds_dev)} samples, {ds_dev.num_speakers} speakers')

# Map dev speakers into the train speaker index space (same 20 speakers,
# possibly different ordering). Drop any dev speaker not in train (should
# be 0; both splits use the same 20 VKTC speakers).
spk_train2idx = ds_train.speaker_to_id
missing_spk = [s for s in ds_dev.speaker_to_id if s not in spk_train2idx]
if missing_spk:
    print(f'  WARN: {len(missing_spk)} dev speakers not in train: {missing_spk[:5]}')
for s in ds_dev.samples:
    s['speaker_id'] = spk_train2idx.get(s['speaker'], -1)

## 5. Run the Probe Across Configs

In [ ]:
def run_probe(train_X, train_y, dev_X, dev_y):
    """Standardize then fit a multinomial logistic-regression probe.

    Returns dict with top-1, top-5 accuracy and chance level.
    """
    # Drop any unmapped dev samples.
    keep = dev_y >= 0
    dev_X, dev_y = dev_X[keep], dev_y[keep]

    scaler = StandardScaler().fit(train_X)
    train_Xs = scaler.transform(train_X)
    dev_Xs   = scaler.transform(dev_X)

    clf = LogisticRegression(
        max_iter=2000, multi_class='multinomial',
        solver='lbfgs', n_jobs=-1, C=1.0,
    ).fit(train_Xs, train_y)

    top1 = accuracy_score(dev_y, clf.predict(dev_Xs))
    proba = clf.predict_proba(dev_Xs)
    n_cls = proba.shape[1]
    top5 = top_k_accuracy_score(dev_y, proba, k=min(5, n_cls), labels=clf.classes_)
    chance = 1.0 / n_cls
    return {'top1': top1, 'top5': top5, 'chance': chance, 'n_classes': n_cls}

results = []
for label, ckpt_path in CHECKPOINTS:
    if not os.path.exists(ckpt_path):
        print(f'[skip] {label}: checkpoint missing at {ckpt_path}')
        continue
    print(f'\n=== {label} ===')
    t0 = time.time()
    model, _ = load_model(ckpt_path)

    tr_X, tr_spk, _ = cached_extract(label, 'train', model, ds_train)
    dv_X, dv_spk, _ = cached_extract(label, 'dev',   model, ds_dev)

    # Free the GPU before the next checkpoint.
    del model
    torch.cuda.empty_cache()

    res = run_probe(tr_X, tr_spk, dv_X, dv_spk)
    res['label'] = label
    res['elapsed_sec'] = time.time() - t0
    results.append(res)
    print(f'  top-1={res["top1"]*100:.2f}%   top-5={res["top5"]*100:.2f}%   '
          f'chance={res["chance"]*100:.2f}%   n_classes={res["n_classes"]}   '
          f'({res["elapsed_sec"]:.0f}s)')

## 6. Summary Table

Read this off into the defense backup slide. The headline number for Belman's question is the gap between `full_glada` top-1 and `backbone_only` top-1: that quantifies what the speaker GRL actually removed.

In [ ]:
if results:
    print(f'\n{"config":<18}{"top-1 (%)":>12}{"top-5 (%)":>12}{"chance (%)":>14}')
    print('-' * 56)
    for r in results:
        print(f'{r["label"]:<18}{r["top1"]*100:>12.2f}{r["top5"]*100:>12.2f}{r["chance"]*100:>14.2f}')

    # If both endpoints are present, report the GRL effect.
    by_label = {r['label']: r for r in results}
    if 'full_glada' in by_label and 'backbone_only' in by_label:
        delta = (by_label['backbone_only']['top1'] - by_label['full_glada']['top1']) * 100
        print(f'\nSpeaker-info reduction by speaker-GRL '
              f'(backbone_only - full_glada): {delta:+.2f} pp top-1')

    # Persist for the slide.
    out_path = f'{CHECKPOINT_ROOT}/speaker_leakage_results.json'
    with open(out_path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f'\nresults written to {out_path}')
else:
    print('no results -- check CHECKPOINTS paths above')